# bob_generated_code.ipynb

Lab notebook for IBM SkillsBuild **AI in Space** (`04_ai_in_space`). Tasks 1–8 use the lab fallback implementations; Task 9 is the completed go/no-go dashboard.


## Task 1: Install required Python libraries


In [ ]:
%pip install -q pandas numpy scikit-learn matplotlib streamlit joblib requests

import pandas as pd
import numpy as np
import sklearn
import matplotlib
import streamlit
import joblib
import requests

print(f"pandas:       {pd.__version__}")
print(f"numpy:        {np.__version__}")
print(f"scikit-learn: {sklearn.__version__}")
print(f"matplotlib:   {matplotlib.__version__}")
print(f"streamlit:    {streamlit.__version__}")
print(f"joblib:       {joblib.__version__}")
print(f"requests:     {requests.__version__}")

## Task 2: Download the dataset


In [ ]:
from pathlib import Path
import requests
import pandas as pd

data_dir = Path("data")
data_dir.mkdir(exist_ok=True)

csv_path = data_dir / "space_weather_unified.csv"
url = "LINKPLACEHOLDER"

if not csv_path.exists():
    print(f"Downloading {url} ...")
    r = requests.get(url, timeout=30)
    r.raise_for_status()
    csv_path.write_bytes(r.content)
    print(f"Saved {csv_path.stat().st_size / 1024:.1f} KB to {csv_path}")
else:
    print(f"Using cached {csv_path}")

matches = pd.read_csv(csv_path, parse_dates=["date"])
print(f"Shape: {matches.shape}")
print(f"Date range: {matches['date'].min().date()} → {matches['date'].max().date()}")
matches.head(3)

## Task 3: Prepare the data


In [ ]:
import pandas as pd
import numpy as np

# Load the dataset
space_df = pd.read_csv('data/space_weather_unified.csv')

print("=== Initial Data Info ===")
print(f"Shape: {space_df.shape}")
print(f"\nColumns and Data Types:")
print(space_df.dtypes.to_string())
print(f"\nFirst 3 rows:")
print(space_df.head(3).to_string())

print("\n=== Missing Values Analysis ===")
missing_counts = space_df.isnull().sum()
missing_pct = (missing_counts / len(space_df) * 100).round(2)
missing_df = pd.DataFrame({
    'Missing_Count': missing_counts,
    'Percentage': missing_pct
})
missing_df = missing_df[missing_df['Missing_Count'] > 0].sort_values('Missing_Count', ascending=False)
print(missing_df.to_string())

print("\n=== Data Cleaning ===")
initial_rows = len(space_df)

# Handle missing values
space_df['kp_index'] = space_df['kp_index'].fillna(0.0)
space_df['class_type'] = space_df['class_type'].fillna('Unknown')
space_df['source_location'] = space_df['source_location'].fillna('Unknown')
space_df['active_region'] = space_df['active_region'].fillna('Unknown')
space_df['note'] = space_df['note'].fillna('')

# Remove duplicates
duplicates_count = space_df.duplicated(subset=['event_id']).sum()
space_df = space_df.drop_duplicates(subset=['event_id'], keep='first')

# Convert datetime columns
datetime_cols = ['begin_time', 'peak_time', 'end_time', 'observed_time']
for col in datetime_cols:
    space_df[col] = pd.to_datetime(space_df[col], errors='coerce')

# Extract year, month, hour from begin_time
space_df['year']  = space_df['begin_time'].dt.year.fillna(0).astype(int)
space_df['month'] = space_df['begin_time'].dt.month.fillna(0).astype(int)
space_df['hour']  = space_df['begin_time'].dt.hour.fillna(0).astype(int)

# Calculate duration in minutes
space_df['duration_minutes'] = (space_df['end_time'] - space_df['begin_time']).dt.total_seconds() / 60
space_df['duration_minutes'] = space_df['duration_minutes'].fillna(0.0)

# Extract flare magnitude and class
def extract_flare_info(row):
    if row['event_type'] == 'Solar Flare' and pd.notna(row['class_type']) and row['class_type'] != 'Unknown':
        class_str = str(row['class_type'])
        # Extract letter class (first character)
        flare_class = class_str[0] if class_str else 'N/A'
        # Extract numeric magnitude
        try:
            magnitude_str = ''.join(c for c in class_str if c.isdigit() or c == '.')
            magnitude = float(magnitude_str) if magnitude_str else 0.0
        except:
            magnitude = 0.0
        return pd.Series({'flare_magnitude': magnitude, 'flare_class': flare_class})
    return pd.Series({'flare_magnitude': 0.0, 'flare_class': 'N/A'})

space_df[['flare_magnitude', 'flare_class']] = space_df.apply(extract_flare_info, axis=1)

final_rows = len(space_df)

print(f"Initial rows: {initial_rows:,}")
print(f"Duplicates removed: {duplicates_count:,}")
print(f"Final rows: {final_rows:,}")
print(f"\nRemaining missing values:")
remaining_missing = space_df.isnull().sum()
remaining_missing = remaining_missing[remaining_missing > 0]
if len(remaining_missing) > 0:
    print(remaining_missing.to_string())
else:
    print("No missing values in key columns (kp_index, class_type, source_location, active_region, note)")

print("\n=== First 3 Rows After Cleaning ===")
space_df.head(3)

## Task 4: Explore and analyze the data


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# === PART A: STATISTICS ===

print("=== 1. Event Type Distribution ===")
event_dist = space_df['event_type'].value_counts()
event_pct = (event_dist / len(space_df) * 100).round(2)
print(pd.DataFrame({'Count': event_dist, 'Percentage': event_pct}).to_string())

print("\n=== 2. Temporal Summary ===")
print("Events per Year:")
print(space_df['year'].value_counts().sort_index().to_string())
print("\nEvents per Month:")
print(space_df['month'].value_counts().sort_index().to_string())
print("\nEvents per Hour:")
print(space_df['hour'].value_counts().sort_index().to_string())

print("\n=== 3. Solar Flare Summary ===")
solar_flares = space_df[space_df['event_type'] == 'Solar Flare']
print(f"Total Solar Flares: {len(solar_flares):,}")
print("\nFlare Class Distribution:")
print(solar_flares['flare_class'].value_counts().to_string())
flare_mag = solar_flares[solar_flares['flare_magnitude'] > 0]['flare_magnitude']
print(f"\nFlare Magnitude — Mean: {flare_mag.mean():.2f}, Median: {flare_mag.median():.2f}, Min: {flare_mag.min():.2f}, Max: {flare_mag.max():.2f}")
flare_dur = solar_flares[solar_flares['duration_minutes'] > 0]['duration_minutes']
print(f"Duration (min)   — Mean: {flare_dur.mean():.2f}, Median: {flare_dur.median():.2f}, Min: {flare_dur.min():.2f}, Max: {flare_dur.max():.2f}")

print("\n=== 4. Geomagnetic Storm Summary ===")
geo_storms = space_df[space_df['event_type'] == 'Geomagnetic Storm']
print(f"Total Geomagnetic Storms: {len(geo_storms):,}")
print("\nStorm Class Distribution:")
print(geo_storms['class_type'].value_counts().to_string())
kp_values = geo_storms[geo_storms['kp_index'] > 0]['kp_index']
if len(kp_values) > 0:
    print(f"\nKp Index — Mean: {kp_values.mean():.2f}, Median: {kp_values.median():.2f}, Min: {kp_values.min():.2f}, Max: {kp_values.max():.2f}")
else:
    print("No Kp index data available")

# === PART B: CHARTS ===

fig, axes = plt.subplots(2, 2, figsize=(14, 10))
fig.suptitle('Exploratory Data Analysis', fontsize=15, fontweight='bold')

# Subplot 1: Event Type Distribution
ax1 = axes[0, 0]
et_counts = space_df['event_type'].value_counts().sort_values(ascending=True)
bars1 = ax1.barh(et_counts.index, et_counts.values, color='#7c5cd8')
for bar, val in zip(bars1, et_counts.values):
    ax1.text(bar.get_width() + et_counts.values.max() * 0.01,
             bar.get_y() + bar.get_height() / 2,
             f'{val:,}', va='center', fontsize=9)
ax1.set_title('Event Type Distribution')
ax1.set_xlabel('Count')

# Subplot 2: Events per Month
ax2 = axes[0, 1]
monthly = space_df['month'].value_counts().sort_index()
ax2.bar(monthly.index, monthly.values, color='#3b82d4')
ax2.set_xticks(range(1, 13))
ax2.yaxis.grid(True, alpha=0.3)
ax2.set_title('Events per Month')
ax2.set_xlabel('Month')
ax2.set_ylabel('Count')

# Subplot 3: Solar Flare Class Breakdown
ax3 = axes[1, 0]
flare_df = space_df[(space_df['event_type'] == 'Solar Flare') & (space_df['flare_class'] != 'N/A')]
fc_counts = flare_df['flare_class'].value_counts().sort_index()
class_colors = {'X': '#dc2626', 'M': '#f97316', 'C': '#eab308', 'B': '#22c55e'}
colors3 = [class_colors.get(c, '#94a3b8') for c in fc_counts.index]
bars3 = ax3.bar(fc_counts.index, fc_counts.values, color=colors3)
for bar, val in zip(bars3, fc_counts.values):
    ax3.text(bar.get_x() + bar.get_width() / 2,
             bar.get_height() + fc_counts.values.max() * 0.01,
             f'{val:,}', ha='center', fontsize=9)
ax3.set_title('Solar Flare Class Breakdown')
ax3.set_xlabel('Flare Class')
ax3.set_ylabel('Count')

# Subplot 4: Events per Year
ax4 = axes[1, 1]
yearly = space_df['year'].value_counts().sort_index()
bars4 = ax4.bar(yearly.index.astype(str), yearly.values, color='#10b981')
for bar, val in zip(bars4, yearly.values):
    ax4.text(bar.get_x() + bar.get_width() / 2,
             bar.get_height() + yearly.values.max() * 0.01,
             f'{val:,}', ha='center', fontsize=9)
ax4.set_title('Events per Year')
ax4.set_xlabel('Year')
ax4.set_ylabel('Count')

plt.tight_layout()
plt.show()

## Task 5: Add feature engineering for risk assessment


In [ ]:
import pandas as pd
from datetime import timedelta

# Filter data from 2023 onward
sw = space_df[space_df['date'] >= '2023-01-01'].sort_values('date').reset_index(drop=True)

# Convert date column to datetime if not already
sw['date'] = pd.to_datetime(sw['date'])

# Get unique dates
unique_dates = sw['date'].unique()
rows = []

for current_date in unique_dates:
    # Get events in last 48 hours
    window_start = current_date - timedelta(hours=48)
    window_data = sw[(sw['date'] >= window_start) & (sw['date'] < current_date)]

    # Count flare classes
    xclass = len(window_data[(window_data['event_type'] == 'Solar Flare') & (window_data['flare_class'] == 'X')])
    mclass = len(window_data[(window_data['event_type'] == 'Solar Flare') & (window_data['flare_class'] == 'M')])
    cclass = len(window_data[(window_data['event_type'] == 'Solar Flare') & (window_data['flare_class'] == 'C')])

    # Kp index statistics
    kp_values = window_data['kp_index'].dropna()
    max_kp = kp_values.max() if len(kp_values) > 0 else 0
    avg_kp = kp_values.mean() if len(kp_values) > 0 else 0

    # Storm count (Kp >= 5)
    storm_count = len(window_data[window_data['kp_index'] >= 5])

    # Event trend (last 24h vs previous 24h)
    last_24h = sw[(sw['date'] >= current_date - timedelta(hours=24)) & (sw['date'] < current_date)]
    prev_24h = sw[(sw['date'] >= current_date - timedelta(hours=48)) & (sw['date'] < current_date - timedelta(hours=24))]
    trend = len(last_24h) / len(prev_24h) if len(prev_24h) > 0 else 1.0

    rows.append({
        'date': current_date,
        'xclass_flare_count': xclass,
        'mclass_flare_count': mclass,
        'cclass_flare_count': cclass,
        'max_kp_index': max_kp,
        'avg_kp_index': avg_kp,
        'storm_count': storm_count,
        'event_trend': trend
    })

risk_features_df = pd.DataFrame(rows)
print(f"Shape: {risk_features_df.shape}")
risk_features_df.head(5)

## Task 6: Create risk scoring model


In [ ]:
def calculate_risk_score(row):
    # X-class contribution (max 40)
    x_score = min(row['xclass_flare_count'] * 40, 40)

    # M-class contribution (max 25)
    m_score = min(row['mclass_flare_count'] * 25, 25)

    # Kp index contribution (max 20)
    kp_score = (row['max_kp_index'] / 9) * 20

    # Trend contribution (max 15)
    trend_score = max((row['event_trend'] - 1) * 15, 0)
    trend_score = min(trend_score, 15)

    total = x_score + m_score + kp_score + trend_score
    return min(total, 100)

# Apply risk scoring
risk_features_df['risk_score'] = risk_features_df.apply(calculate_risk_score, axis=1)

# Categorize risk levels
def categorize_risk(score):
    if score <= 20:
        return 'LOW'
    elif score <= 40:
        return 'MODERATE'
    elif score <= 60:
        return 'HIGH'
    else:
        return 'EXTREME'

risk_features_df['risk_level'] = risk_features_df['risk_score'].apply(categorize_risk)

# Print statistics
print("Risk Level Distribution:")
print(risk_features_df['risk_level'].value_counts().to_string())

print("\nRisk Score Statistics:")
print(f"Mean: {risk_features_df['risk_score'].mean():.2f}")
print(f"Median: {risk_features_df['risk_score'].median():.2f}")
print(f"Min: {risk_features_df['risk_score'].min():.2f}")
print(f"Max: {risk_features_df['risk_score'].max():.2f}")

print("\nTop 5 Highest Risk Dates:")
print(risk_features_df.nlargest(5, 'risk_score')[['date', 'risk_score', 'risk_level']].to_string())

risk_features_df.head(10)

## Task 7: Train and evaluate decision model


In [ ]:
import pandas as pd
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report

# Define features
feature_cols = ['xclass_flare_count', 'mclass_flare_count', 'cclass_flare_count',
                'max_kp_index', 'avg_kp_index', 'storm_count', 'event_trend']

# Time-based split
cutoff = pd.Timestamp('2025-01-01')
train_mask = risk_features_df['date'] < cutoff
test_mask = risk_features_df['date'] >= cutoff

X_train = risk_features_df.loc[train_mask, feature_cols]
X_test = risk_features_df.loc[test_mask, feature_cols]
y_train = risk_features_df.loc[train_mask, 'risk_level']
y_test = risk_features_df.loc[test_mask, 'risk_level']

print(f"X_train: {X_train.shape}, X_test: {X_test.shape}")
print(f"y_train: {y_train.shape}, y_test: {y_test.shape}")

# Train model
model = RandomForestClassifier(n_estimators=100, max_depth=10, random_state=42, n_jobs=-1)
model.fit(X_train, y_train)
y_pred = model.predict(X_test)

# Evaluate
acc = accuracy_score(y_test, y_pred)
print(f"\nTest accuracy: {acc*100:.2f}%")

print("\nClassification Report:")
print(classification_report(y_test, y_pred))

print("\nFeature Importances:")
for name, imp in sorted(zip(feature_cols, model.feature_importances_), key=lambda x: -x[1]):
    print(f"  {name:<25} {imp:.4f}")

## Task 8: Save model and risk data


In [ ]:
from pathlib import Path
import joblib

# Create models directory
models_dir = Path("models")
models_dir.mkdir(exist_ok=True)

# Compute current statistics
latest_row = risk_features_df.iloc[-1]
current_stats = {
    'latest_date': latest_row['date'],
    'latest_risk_score': latest_row['risk_score'],
    'latest_risk_level': latest_row['risk_level'],
    'xclass_48h': int(latest_row['xclass_flare_count']),
    'mclass_48h': int(latest_row['mclass_flare_count']),
    'max_kp_48h': float(latest_row['max_kp_index']),
    'recommendation': 'GO' if latest_row['risk_level'] == 'LOW' else
                     'CAUTION' if latest_row['risk_level'] == 'MODERATE' else
                     'DELAY' if latest_row['risk_level'] == 'HIGH' else 'NO-GO'
}

# Save model
joblib.dump(model, models_dir / "launch_decision_model.pkl")

# Save data
data_package = {
    'current_stats': current_stats,
    'feature_cols': feature_cols,
    'risk_features': risk_features_df.tail(30).to_dict('records')
}
joblib.dump(data_package, models_dir / "space_weather_data.pkl")

# Print summary
print("=" * 60)
print("SPACE WEATHER LAUNCH SAFETY SYSTEM")
print("=" * 60)
print(f"\nCurrent Date: {current_stats['latest_date']}")
print(f"Risk Score: {current_stats['latest_risk_score']:.1f}/100")
print(f"Risk Level: {current_stats['latest_risk_level']}")
print(f"\n🚀 LAUNCH RECOMMENDATION: {current_stats['recommendation']}")
print("\nKey Metrics (Last 48 hours):")
print(f"  X-class flares: {current_stats['xclass_48h']}")
print(f"  M-class flares: {current_stats['mclass_48h']}")
print(f"  Max Kp index: {current_stats['max_kp_48h']:.1f}")
print("\n✅ Model saved to: models/launch_decision_model.pkl")
print("✅ Data saved to: models/space_weather_data.pkl")

## Task 9: Build a date-range go/no-go dashboard


In [ ]:
import joblib
import pandas as pd
import matplotlib.pyplot as plt

# ── Edit these two variables to change the analysis window ─────────────────
START_DATE = '2025-05-01'
END_DATE   = '2025-07-09'
# ───────────────────────────────────────────────────────────────────────────

# Load saved data
pkg = joblib.load('models/space_weather_data.pkl')
dash_df = pd.DataFrame(pkg['risk_features'])
dash_df['date'] = pd.to_datetime(dash_df['date'])

# Filter to selected window
window_df = dash_df[
    (dash_df['date'] >= pd.Timestamp(START_DATE)) &
    (dash_df['date'] <= pd.Timestamp(END_DATE))
].reset_index(drop=True)

# Compute window summary
rec_map = {'LOW': 'GO', 'MODERATE': 'CAUTION', 'HIGH': 'DELAY', 'EXTREME': 'NO-GO'}
dominant_level = window_df['risk_level'].mode()[0] if len(window_df) else 'LOW'
max_idx = window_df['risk_score'].idxmax() if len(window_df) else 0
window_summary = {
    'total_days':             len(window_df),
    'avg_risk_score':         round(window_df['risk_score'].mean(), 2) if len(window_df) else 0,
    'go_days':                int((window_df['risk_level'] == 'LOW').sum()),
    'caution_days':           int((window_df['risk_level'] == 'MODERATE').sum()),
    'delay_days':             int((window_df['risk_level'] == 'HIGH').sum()),
    'nogo_days':              int((window_df['risk_level'] == 'EXTREME').sum()),
    'overall_recommendation': rec_map[dominant_level],
    'max_risk_score':         float(window_df['risk_score'].max()) if len(window_df) else 0,
    'max_risk_date':          str(window_df.loc[max_idx, 'date'])[:10] if len(window_df) else 'N/A',
}

# Print summary
print(f"=== Go/No-Go Analysis: {START_DATE} to {END_DATE} ===")
print(f"Overall Recommendation : {window_summary['overall_recommendation']}")
print(f"Days analysed          : {window_summary['total_days']}")
print(f"Avg risk score         : {window_summary['avg_risk_score']:.2f} / 100")
print(f"GO days                : {window_summary['go_days']}")
print(f"CAUTION days           : {window_summary['caution_days']}")
print(f"DELAY days             : {window_summary['delay_days']}")
print(f"NO-GO days             : {window_summary['nogo_days']}")
print(f"Highest risk date      : {window_summary['max_risk_date']}  (score {window_summary['max_risk_score']:.1f})")

if len(window_df) == 0:
    print('\nNo data in selected window.')
else:
    color_map = {'LOW': '#22c55e', 'MODERATE': '#f59e0b', 'HIGH': '#f97316', 'EXTREME': '#dc2626'}
    bar_colors = window_df['risk_level'].map(color_map).tolist()
    dates = window_df['date'].dt.strftime('%Y-%m-%d').tolist()

    fig, axes = plt.subplots(3, 1, figsize=(14, 10))
    fig.suptitle(
        f'Space Weather Go/No-Go Dashboard  |  {START_DATE} to {END_DATE}',
        fontsize=14, fontweight='bold'
    )

    # Subplot 1: Risk Score Trend
    ax1 = axes[0]
    ax1.bar(dates, window_df['risk_score'], color=bar_colors, width=0.6)
    ax1.axhline(20, color='#57606a', linestyle='--', linewidth=1, label='GO threshold (20)')
    ax1.axhline(60, color='#57606a', linestyle=':',  linewidth=1, label='NO-GO threshold (60)')
    ax1.set_title('Risk Score per Day')
    ax1.set_ylabel('Risk Score (0-100)')
    ax1.set_ylim(0, 105)
    ax1.set_xticks(range(len(dates)))
    ax1.set_xticklabels(dates, rotation=45, ha='right', fontsize=8)
    ax1.legend(fontsize=8)
    ax1.yaxis.grid(True, alpha=0.3)

    # Subplot 2: Daily Recommendation
    ax2 = axes[1]
    level_num = window_df['risk_level'].map({'LOW': 1, 'MODERATE': 2, 'HIGH': 3, 'EXTREME': 4}).tolist()
    ax2.bar(dates, level_num, color=bar_colors, width=0.6)
    ax2.set_yticks([1, 2, 3, 4])
    ax2.set_yticklabels(['GO', 'CAUTION', 'DELAY', 'NO-GO'])
    ax2.set_title('Daily Recommendation')
    ax2.set_ylim(0, 5)
    ax2.set_xticks(range(len(dates)))
    ax2.set_xticklabels(dates, rotation=45, ha='right', fontsize=8)
    ax2.yaxis.grid(True, alpha=0.3)

    # Subplot 3: Solar Event Counts (stacked)
    ax3 = axes[2]
    ax3.bar(dates, window_df['xclass_flare_count'], color='#dc2626', width=0.6, label='X-class')
    ax3.bar(dates, window_df['mclass_flare_count'], color='#f97316', width=0.6, label='M-class',
            bottom=window_df['xclass_flare_count'])
    ax3.bar(dates, window_df['cclass_flare_count'], color='#eab308', width=0.6, label='C-class',
            bottom=window_df['xclass_flare_count'] + window_df['mclass_flare_count'])
    ax3.bar(dates, window_df['storm_count'], color='#7c5cd8', width=0.6, label='Storms',
            bottom=window_df['xclass_flare_count'] + window_df['mclass_flare_count'] + window_df['cclass_flare_count'])
    ax3.set_title('Solar Events in 48h Window')
    ax3.set_xlabel('Date')
    ax3.set_ylabel('Event Count')
    ax3.set_xticks(range(len(dates)))
    ax3.set_xticklabels(dates, rotation=45, ha='right', fontsize=8)
    ax3.legend(fontsize=8)
    ax3.yaxis.grid(True, alpha=0.3)

    plt.tight_layout()
    plt.savefig('models/go_nogo_dashboard.png', dpi=120, bbox_inches='tight')
    plt.show()
    print('Saved chart: models/go_nogo_dashboard.png')

